# S06-13A (Student) — Building Graph from Rhino OBJ

Model the building in **Rhino**. Export **four OBJ files**:

| File | Layer |
|------|--------|
| `ground.obj` | slab / podium |
| `columns.obj` | columns |
| `offices.obj` | office volumes |
| `core.obj` | core + corridors |

Store them in `Supporting Files/` (or change `SUPPORT_DIR` in the setup cell).

---

## Pipeline

1. **Import** OBJs → `Topology.ByOBJPath`
2. **Tag** each cell (`cell_type` selectors)
3. **CellComplex** → merge all cells (`Topology.SelfMerge`)
4. **Transfer** dictionaries onto cells
5. **Graph** → `Graph.ByTopology` + one-hot features
6. **Export** CSV → `Graph.ExportToCSV`
7. **Predict** → **S06-13** (Phase 2), `pyg_model.pt`


Reference outputs (gray) below = example building; yours will differ.


### Setup

Install `topologicpy` if needed. Import `Topology`, `Cluster`, `Dictionary`, `Graph`, `Helper`.

In [15]:

# !pip install topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper

SUPPORT_DIR = r"C:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\Assignement03"


### 1. Import parts

`Topology.ByOBJPath` for each OBJ. **TODO:** your four paths.

In [16]:
OBJ_DIR = r"C:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\Assignement03"

ground_objs = Topology.ByOBJPath(OBJ_DIR + r"\ground.obj", transposeAxes=False)
column_objs = Topology.ByOBJPath(OBJ_DIR + r"\columns.obj", transposeAxes=False)
office_objs = Topology.ByOBJPath(OBJ_DIR + r"\offices.obj", transposeAxes=False)
core_objs   = Topology.ByOBJPath(OBJ_DIR + r"\core.obj",    transposeAxes=False)

print("ground:", ground_objs)
print("columns:", column_objs)
print("offices:", office_objs)
print("core:", core_objs)

ground: [<topologic_core.Cluster object at 0x00000220B758AE30>, <topologic_core.Cluster object at 0x00000220B758A4B0>]
columns: [<topologic_core.Cluster object at 0x00000220B7585470>, <topologic_core.Cluster object at 0x00000220B75852F0>, <topologic_core.Cluster object at 0x00000220B7585C70>, <topologic_core.Cluster object at 0x00000220B75865B0>, <topologic_core.Cluster object at 0x00000220B7584EF0>, <topologic_core.Cluster object at 0x00000220B7586170>, <topologic_core.Cluster object at 0x00000220B7585AF0>, <topologic_core.Cluster object at 0x00000220B7585970>]
offices: [<topologic_core.Cluster object at 0x00000220B7321570>, <topologic_core.Cluster object at 0x00000220B73231F0>, <topologic_core.Cluster object at 0x00000220B73231B0>, <topologic_core.Cluster object at 0x00000220B7322FF0>, <topologic_core.Cluster object at 0x00000220B73224F0>, <topologic_core.Cluster object at 0x00000220B7323070>, <topologic_core.Cluster object at 0x00000220B73213B0>, <topologic_core.Cluster object at 0x

**Check:** `Topology.Show(ground_objs, office_objs, core_objs, column_objs)`

In [17]:
Topology.Show(ground_objs, office_objs, core_objs, column_objs, renderer="vscode")

### 2. Assign categories

Per layer: Faces → Flatten → SelfMerge → `Topology.Cells`.  
For each cell: `Dictionary` with `cell_type`, `cell_name`, `cell_color`; `InternalVertex` + `SetDictionary` → `selectors`.

In [18]:
selectors = []

# Plinth (cell_type = 1) — the raised podium slab the columns stand on
ground_faces = [Topology.Faces(obj) for obj in ground_objs if Topology.IsInstance(obj, "Topology")]
ground_faces = Helper.Flatten(ground_faces)
ground = Topology.SelfMerge(Cluster.ByTopologies(ground_faces))
ground_cells = Topology.Cells(ground)
for cell in ground_cells:
    d = Dictionary.ByKeysValues(["cell_type", "cell_name", "cell_color"], [1, "plinth", "darkslategray"])
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

# Columns (cell_type = 2) — pilotis
column_faces = [Topology.Faces(obj) for obj in column_objs if Topology.IsInstance(obj, "Topology")]
column_faces = Helper.Flatten(column_faces)
columns = Topology.SelfMerge(Cluster.ByTopologies(column_faces))
column_cells = Topology.Cells(columns)
for cell in column_cells:
    d = Dictionary.ByKeysValues(["cell_type", "cell_name", "cell_color"], [2, "column", "lightsteelblue"])
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

# Offices / habitable volumes (cell_type = 3)
office_faces = [Topology.Faces(obj) for obj in office_objs if Topology.IsInstance(obj, "Topology")]
office_faces = Helper.Flatten(office_faces)
offices = Topology.SelfMerge(Cluster.ByTopologies(office_faces))
office_cells = Topology.Cells(offices)
for cell in office_cells:
    d = Dictionary.ByKeysValues(["cell_type", "cell_name", "cell_color"], [3, "office", "burlywood"])
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

# Core / stairs (cell_type = 4)
core_faces = [Topology.Faces(obj) for obj in core_objs if Topology.IsInstance(obj, "Topology")]
core_faces = Helper.Flatten(core_faces)
core = Topology.SelfMerge(Cluster.ByTopologies(core_faces))
core_cells = Topology.Cells(core)
for cell in core_cells:
    d = Dictionary.ByKeysValues(["cell_type", "cell_name", "cell_color"], [4, "core", "sienna"])
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

print(f"Total selectors: {len(selectors)}")

Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: <module>
Total selectors: 22


### 3. CellComplex

`all_cells` → `model = Topology.SelfMerge(Cluster.ByTopologies(all_cells))`

In [19]:
all_cells = ground_cells + column_cells + office_cells + core_cells
model = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
print("Model:", model)
print("Number of cells:", len(Topology.Cells(model)))
Topology.Show(Topology.Cells(model), faceColorKey="cell_color", backgroundColor="white", edgeColor="pink", renderer="vscode")

Model: <topologic_core.CellComplex object at 0x00000220B62261B0>
Number of cells: 31


### 4. Transfer dictionaries

`Topology.TransferDictionariesBySelectors(model, selectors, tranCells=True)`

In [20]:
model = Topology.TransferDictionariesBySelectors(model, selectors, tranCells=True)
Topology.Show(Topology.Cells(model), faceColorKey="cell_color", faceOpacity=1,  backgroundColor="white", edgeColor="white", width=800, height=600, edgeWidth=3, renderer="vscode")

### 5. Adjacency graph

`Graph.ByTopology(model)` + one-hot `feature_00`…`feature_04` from `cell_type`.

In [21]:
def one_hot_encode(value, n):
    if value < 0 or value >= n:
        raise ValueError(f"value {value} is out of range [0, {n-1}]")
    result = [0] * n
    result[value] = 1
    return result

graph = Graph.ByTopology(model)
vertices = Graph.Vertices(graph)

feature_names = []
for i in range(5):
    feature_names.append("feature_" + str(i).zfill(2))

print(feature_names)

skipped = 0
for v in vertices:
    d = Topology.Dictionary(v)
    cell_type = Dictionary.ValueAtKey(d, "cell_type")
    if cell_type is None:
        skipped += 1
        continue
    ohe = one_hot_encode(cell_type, 5)
    for i, feature_name in enumerate(feature_names):
        d = Dictionary.SetValueAtKey(d, feature_name, ohe[i])
    v = Topology.SetDictionary(v, d)

print(f"Vertices processed: {len(vertices) - skipped}, skipped (no cell_type): {skipped}")

for v in vertices:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: _append_edge
DEBUG e (Edge): None
DEBUG - src: 1
DEBUG - dst: 2
DEBUG - gv1 coordinates: [2.9645, 1.8514950000000003, 1.1]
DEBUG - gv2 coordinates: [2.9645, 1.851495, 1.1]
Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: _append_edge
DEBUG e (Edge): None
DEBUG - src: 3
DEBUG - dst: 4
DEBUG - gv1 coordinates: [5.714500000000001, 1.8514950000000001, 1.1]
DEBUG - gv2 coordinates: [5.7145, 1.8514950000000001, 1.1000000000000003]
Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: _append_edge
DEBUG e (Edge): None
DEBUG - src: 5
DEBUG - dst: 6
DEBUG - gv1 coordinates: [8.464499999999997, 1.851495, 1.1]
DEBUG - gv

In [22]:
Graph.Show(graph, vertexSize=20, vertexColorKey="cell_color", vertexLabelKey="cell_name",backgroundColor="white", edgeColor="grey", width=800, height=600, edgeWidth=3, renderer="vscode")

### 6. Export CSV

`Graph.ExportToCSV` → folder with `graphs.csv`, `nodes.csv`, `edges.csv`.

In [24]:
import pandas as pd

EXPORT_DIR = r"C:\Users\Win11\GraphML_RaniaChihaoui\Exports"

status = Graph.ExportToCSV(graph,
                           path=EXPORT_DIR,
                           nodeFeaturesKeys=feature_names,
                           nodeLabelKey="cell_type",
                           overwrite=True)
print(status)

# Fix the graph label to 1 (Separation with Plinth)
graphs_path = EXPORT_DIR + r"\graphs.csv"
graphs_df = pd.read_csv(graphs_path)
graphs_df["label"] = 1
graphs_df.to_csv(graphs_path, index=False)
print("Graph label updated to 1 — Separation with Plinth")

True
Graph label updated to 1 — Separation with Plinth


### 7. Predict (S06-13)

**S06-13 GML Graph Classification** → Phase 2: set `dataset_dir` to your export folder, `LoadModel(pyg_model.pt)`, `Predict()` → label 0–4.